In [7]:
# from flask import Flask, request
# from flask_cors import CORS
import numpy as np
# import tabix
import sys
import os
# from git import Repo
import json
# import h5py
from subprocess import Popen, PIPE
# from simsearch_functions import simsearch, roiURL  # , load_per

**Develop a method for building a query by including factors**

In [308]:
production = False
RN_VERSION_STR = ''
# load search spaces for each scale
data_dir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'

# Given a scale, build a path name for the search space
def build_search_space_path(scale, method='sliding_window'):
    if method == 'sliding_window':
        path = os.path.join(data_dir, 'sliding_window/regions_{}kb_CT20230427.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.limited_search_space_top_1000_no_overlap.npz'.format(scale))
    elif method == 'hilbert':
        path = os.path.join(data_dir, 'hilbert/hilbert_regions_order_{}_CT20230606.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.limited_search_space_top_1000_no_overlap.npz'.format(scale))
    elif method == 'hilbert_sfc':
        # path = os.path.join(data_dir, 'hilbert_sfc/hilbert_regions_order_{}_CT20230630.dhs.cs.sfc.ranks.limited_search_space_top_1000_no_overlap.npz'.format(scale))
        path = os.path.join(data_dir, 'hilbert_sfc/hilbert_regions_order_{}_CT20230718.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.npz'.format(scale))
    return path


# Given a scale, build a path name for the tabix query
def build_tabix_path(scale, method='sliding_window'):
    if method == 'sliding_window':
        filename = 'sliding_window/hg38.{}kb.regions.CT20230427.sorted.rownum4.factorranks5t38.bed.gz'.format(scale)
    elif method == 'hilbert':
        filename = 'hilbert/hilbert_order_{}_windows.sorted.rownum4.factorranks5t38.bed.gz'.format(scale)
    elif method == 'hilbert_sfc':
        filename = 'hilbert_sfc/20230718/hilbert_order_{}_windows.sorted.rownum4.sfc.ranks5t38.bed.gz'.format(scale)
    if production:
        path = 'http://explore.altius.org/tabix/ng/'
    else:
        path = 'https://resources.altius.org/~ctrader/public/genomic_narration/indexed/'
    return os.path.join(path, filename)


# Load search space data from a given scale
def load_ranks(search_space_path, method='sliding_window'):
    search_space = np.load(search_space_path)
    if ((method == 'sliding_window') | (method == 'hilbert')):
        percents = search_space['percentiles']
    elif (method == 'hilbert_sfc'):
        percents = search_space['ranks']
    coords = search_space['coordinates']
    return percents, coords


# load search spaces for all scales into dict
# sliding_window_seach_spaces = {}
# hilbert_search_spaces = {}
hilbert_sfc_search_spaces = {}
# for scale in [1, 2, 5, 10, 15, 25, 50, 75, 100]:
#     region_method = 'sliding_window'
#     sliding_window_seach_spaces[scale] = load_ranks(build_search_space_path(scale, region_method), region_method)
# for scale in [4, 5, 6, 7, 8, 9, 10, 11]:
#     region_method = 'hilbert'
#     hilbert_search_spaces[scale] = load_ranks(build_search_space_path(scale, region_method), region_method)
for scale in [4, 5, 6, 7, 8, 9, 10, 11]:
    region_method = 'hilbert_sfc'
    hilbert_sfc_search_spaces[scale] = load_ranks(build_search_space_path(scale, region_method), region_method)
    
# choose search space from scale options
def choose_search_space(scale, method='sliding_window'):
    global percents, coords
    if method == 'sliding_window':
        percents, coords = sliding_window_seach_spaces[scale]
    elif method == 'hilbert':
        percents, coords = hilbert_search_spaces[scale]
    elif method == 'hilbert_sfc':
        percents, coords = hilbert_sfc_search_spaces[scale]
        
        
####### new #######
        
# find the factors we want to search over
def setup_selected_factors(selected_factors_str):
    global selected_factors
    selected_factors = np.int_(
        selected_factors_str.strip('[').strip(']').split(',')
    )

def determine_factor_order(top_regions, selected_factors):
    top_region_ranks = percents[top_regions]

    # order selected factors first
    selected_factors_best_order = selected_factors[
        np.argsort(
            -np.sum(
                top_region_ranks[:, selected_factors], 
                axis=0
            )
        )
    ]

    # order remaining factors
    regions_to_consider = np.where(np.count_nonzero(percents[:, selected_factors_best_order], axis=1) > 0)[0]
    total_factors = np.arange(percents.shape[1])
    leftover_factors = total_factors[~np.isin(total_factors, selected_factors_best_order)]
    leftover_factors_best_order = leftover_factors[
        np.argsort(
            -np.sum(
                percents[regions_to_consider][:, leftover_factors], 
                axis=0
            ), 
            kind='stable'
        )
    ]

    factor_order = np.append(selected_factors_best_order, leftover_factors_best_order)
    return factor_order
    
    


# search for the best regions for selected factors
def search_over_selected_factors(num_regions):
    global top_regions, top_regions_ranks, top_regions_coords, top_regions_score, factor_order
    # focus on selected factors
    selected_factor_search_space_ranks = percents[:, selected_factors]
    # fusion
    selected_factor_search_space_score = 1 - np.prod(1 - selected_factor_search_space_ranks, axis=1)
    # get the best regions
    top_regions = np.argsort(-selected_factor_search_space_score)[:num_regions]
    
    # factor order
    factor_order = determine_factor_order(top_regions, selected_factors)
    
    # find info for each top region
    top_regions_ranks = percents[top_regions][:, factor_order]
    top_regions_coords = coords[top_regions]
    top_regions_score = selected_factor_search_space_score[top_regions]


# build array of objects for each region at each detail level
def generate_search_by_factor_result_array(num_regions):
    global search_by_factor_return_arr
    # for each region, return relevant information
    search_by_factor_return_arr = []
    for rank in range(num_regions):
        coord = top_regions_coords[rank]
        region_dict = {
            'rank': rank,
            'coordinates': coord[0] + ':' + coord[1] + '-' + coord[2],
            'region_ind': int(top_regions[rank]),
            'score': float(top_regions_score[rank]),
            'percentiles': top_regions_ranks[rank].tolist()
        }
        if rank == 0:
            region_dict['factor_order'] = factor_order.tolist()
            region_dict['selected_factors'] = selected_factors.tolist()
        search_by_factor_return_arr.append(region_dict)

array([ 3,  1,  2, 10,  0,  7,  6, 12, 33,  5,  8,  4, 13, 14, 15, 11,  9,
       21, 27, 28, 16, 17, 18, 19, 20, 22, 23, 24, 25, 26, 29, 30, 31, 32])

In [297]:
selected_factors_best_order

array([3, 1])

In [298]:
leftover_factors_ordered

array([ 2, 10,  0,  7,  6, 12, 33,  5,  8,  4, 13, 14, 15, 11,  9, 21, 27,
       28, 16, 17, 18, 19, 20, 22, 23, 24, 25, 26, 29, 30, 31, 32])

In [309]:
# information that would be provided

scale = 4
selected_factors_str = '[1,3]'
# get_roiURL='False'
num_regions=20
region_method='hilbert_sfc'

In [310]:
# def search_by_factor():
# request_info = request.get_json()
# scale = request_info['scale']
# selected_factors_str = request_info['factors']
# num_regions = int(request_info['numRegions'])
# region_method = request_info['regionMethod']

# choose search space
choose_search_space(scale, region_method)

# get the selected factors
setup_selected_factors(selected_factors_str)

# get the top regions for the selected factors
search_over_selected_factors(num_regions)

# build return array
generate_search_by_factor_result_array(num_regions)

# return results
json_return = json.dumps(search_by_factor_return_arr)
# return json_return


In [311]:
json_return

'[{"rank": 0, "coordinates": "chrX:61865984-62914560", "region_ind": 2295, "score": 1.0, "percentiles": [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "factor_order": [3, 1, 2, 10, 0, 7, 6, 12, 33, 5, 8, 4, 13, 14, 15, 11, 9, 21, 27, 28, 16, 17, 18, 19, 20, 22, 23, 24, 25, 26, 29, 30, 31, 32], "selected_factors": [1, 3]}, {"rank": 1, "coordinates": "chr18:22020096-23068672", "region_ind": 879, "score": 1.0, "percentiles": [1.0, 0.0, 0.0, 0.7922868728637695, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9979702234268188, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9695534706115723, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}, {"rank": 2, "coordinates": "chr21:4194304-5242880", "region_ind": 1229, "score": 0.9996616840362549, "percentiles": [0.0, 0.9996616840362549, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,